# FEniCSx Navier-Stokes (NS) Solver
---
Technical solver details:
- Taylor-Hood elements for velocity and pressure
- BDF2 time-stepping scheme

Goals for this solver:
- 2D Exact Taylor-Green vortex
    - Compare norms
- 3D Exact solution Steinmann vortex problem
    - Compare norms
- 2D DFG benchmark problem
    - Does solver preform as expected?
- Simple anulus spinal geometry
---

## Taylor-Green Vortex Problem [[1]](https://en.wikipedia.org/wiki/Taylor%E2%80%93Green_vortex)

The original work describes a 3 dimensional flow. It's defined by the three velocity components $\boldsymbol v = (u,v,w) $ at time $t = 0$ governed bt the three equations:
$$
\begin{align*}
u = A \cos(a x )\sin(by)\sin(cz), \\
v = B \sin(ax)\cos(by)\sin(cz), \\
w = C \sin(ax)\sin(by)\cos(cz). \\
\end{align*}
$$

The continuity eq. $\nabla \cdot \boldsymbol v = 0$ determines that $Aa + Bb + Cc = 0$.

An exact solution is known in two spatial dimensions, and it's what we will try to replicate with the FEniCSx NS solver.

### The Incompressible NS eq.
The incompressible Navier-Stokes equations are given by:

$$
\begin{align*}
&\frac{\partial u}{\partial x} + \frac{\partial v}{\partial y} = 0, \\
&\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} + v \frac{\partial u}{\partial y} = - \frac 1 \rho \frac{\partial p}{\partial x} + \nu \left( \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} \right), \\
&\frac{\partial v}{\partial t} + u \frac{\partial v}{\partial x} + v \frac{\partial v}{\partial y} = - \frac 1 \rho \frac{\partial p}{\partial y} + \nu \left( \frac{\partial^2 v}{\partial x^2} + \frac{\partial^2 v}{\partial y^2} \right).
\end{align*}
$$

The first equation is the continuity equation. The two under are the momentum equations.


### Taylor-Green Vortex Exact Solution

In an infinite domain, the doubly periodic solution is given by 

$$
\begin{align*}

u = U_0 \sin(kx)\cos(ky)F(t), & \quad v = -U_0 \cos(kx)\sin(ky)F(t), \\
F(t)&=e^{-\nu k^2t}

\end{align*}
$$

Here $U_0$ is the maximum velocity in the flow field, $k$ is the inverse length scale, $\nu$ is the kinematic viscosity. From Taylor and Green we know that with $A=a=b=1$ we get this exact solution. Further we can expand the exponential as a Taylor series, $F(t) = 1 - 2\nu k^2t + \mathcal O(t^2)$. The pressure field $p$ is obtained by substituting the velocity solution in to the momentum equations, 

$$
\begin{align*}

p = \frac{\rho U_0^2}{4}(\cos(2k x)+ \cos (2 k y)) F(t)^2

\end{align*}
$$

With $\rho$ being the fluid density.

The stream function of the Taylor-Green vortex is given by, that satisfies $\boldsymbol v = \nabla \times \psi$,

$$

\begin{align*}
\psi = U_0 \sin(kx)\sin(ky)F(t) \boldsymbol{\hat z}.
\end{align*}

$$

With the similar voricity given by, that also satisfies $\boldsymbol \omega = \nabla \times \boldsymbol v$,

$$

\begin{align*}
\omega = 2U_0\sin(kx)\sin(ky)F(t)\boldsymbol{\hat z}.   
\end{align*}

$$
---

In [1]:
# Pressure-driven Poiseuille flow on the unit square.

import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem
from dolfinx.fem.petsc import LinearProblem


def walls(x):
       return np.logical_or(np.isclose(x[1], 0.0), np.isclose(x[1], 1.0))

def inflow(x):
       return np.isclose(x[0], 0.0)

def outflow(x):
       return np.isclose(x[0], 1.0)


def solve_ns(msh, rho=1.0, mu=1.0, dt=0.02, t_end=10.0, outlet=None, gamma=0.0, beta=1.0):

    gdim = msh.geometry.dim

    V = fem.functionspace(msh, ("Lagrange", 2, (gdim,))) # P2
    Q = fem.functionspace(msh, ("Lagrange", 1)) # P1
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    p, q = ufl.TrialFunction(Q), ufl.TestFunction(Q)

    u_h = fem.Function(V, name="velocity")
    p_h = fem.Function(Q, name="pressure")
    u_n = fem.Function(V, name="u_n")

    rho_c = fem.Constant(msh, PETSc.ScalarType(rho))
    mu_c = fem.Constant(msh, PETSc.ScalarType(mu))
    dt_c = fem.Constant(msh, PETSc.ScalarType(dt))
    gamma_c = fem.Constant(msh, PETSc.ScalarType(gamma))

    dx = ufl.Measure("dx", msh, metadata={"quadrature_degree": 4})
    eps = lambda z: ufl.sym(ufl.nabla_grad(z))
    convection = rho_c * ufl.inner(ufl.dot(u_n, ufl.nabla_grad(u_n)), v) * dx

    a00 = (rho_c / dt_c * ufl.inner(u, v) * dx
           + 2 * mu_c * ufl.inner(eps(u), eps(v)) * dx
           + gamma_c * ufl.div(u) * ufl.div(v) * dx)
    a01 = -p * ufl.div(v) * dx
    a10 = -q * ufl.div(u) * dx
    a11 = fem.Constant(msh, PETSc.ScalarType(0.0)) * p * q * dx
    L0 = rho_c / dt_c * ufl.inner(u_n, v) * dx - convection
    L1 = fem.Constant(msh, PETSc.ScalarType(0.0)) * q * dx

    wall_dofs = fem.locate_dofs_geometrical(V, walls)
    inflow_dofs = fem.locate_dofs_geometrical(Q, inflow)
    outflow_dofs = fem.locate_dofs_geometrical(Q, outflow)
    zero_velocity = np.zeros(gdim, dtype=PETSc.ScalarType)
    bcs = [
        fem.dirichletbc(zero_velocity, wall_dofs, V),
        fem.dirichletbc(PETSc.ScalarType(8.0), inflow_dofs, Q),
        fem.dirichletbc(PETSc.ScalarType(0.0), outflow_dofs, Q),
    ]

    problem = LinearProblem(
        [[a00, a01], [a10, a11]], [L0, L1], u=[u_h, p_h], bcs=bcs,
        petsc_options_prefix="ns_",
        petsc_options={"ksp_type": "preonly", "pc_type": "lu"},
    )

    num_steps = int(round(t_end / dt))
    history = {"t": [], "change": []}
    t = 0.0
    for step in range(1, num_steps + 1):
        t = step * dt
        problem.solve()
        change = np.linalg.norm(u_h.x.array - u_n.x.array)
        history["t"].append(t)
        history["change"].append(change)
        u_n.x.array[:] = u_h.x.array

    return u_h, p_h, history


In [2]:
from mpi4py import MPI
from dolfinx.mesh import create_unit_square
from dolfinx.plot import vtk_mesh

msh = create_unit_square(MPI.COMM_WORLD, 10, 10)
u_h, p_h, history = solve_ns(msh, rho=1.0, mu=1.0, dt=0.005, t_end=10.0)
print(f"Solved {len(history['t'])} time steps; t = {history['t'][-1]:.2f}")


Solved 2000 time steps; t = 10.00


In [3]:
import pyvista

# Convert the computed P2 velocity field to a VTK grid.
topology, cell_types, geometry = vtk_mesh(u_h.function_space)
gdim = msh.geometry.dim
velocity = u_h.x.array.real.reshape((geometry.shape[0], gdim))
values = np.zeros((geometry.shape[0], 3), dtype=np.float64)
values[:, :gdim] = velocity

function_grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)
function_grid["u"] = values
function_grid["speed"] = np.linalg.norm(velocity, axis=1)
glyphs = function_grid.glyph(orient="u", factor=0.12, tolerance=0.02)

plotter = pyvista.Plotter()
plotter.add_mesh(
    function_grid,
    scalars="speed",
    cmap="viridis",
    show_edges=True,
    scalar_bar_args={"title": "Velocity magnitude"},
)
plotter.add_mesh(glyphs, color="white")
plotter.view_xy()
plotter.add_title("Pressure-driven channel velocity")

if pyvista.OFF_SCREEN:
    plotter.screenshot("channel_velocity.png")
else:
    plotter.show()


Widget(value='<iframe src="http://localhost:39253/index.html?ui=P_0x7a9500166e40_0&reconnect=auto" class="pyvi…